In [7]:
from sklearn.datasets import load_breast_cancer
import numpy as np
data = load_breast_cancer()
X = data.data        # shape (569, 30)
y = data.target      # shape (569,)

In [2]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [3]:
X_train = X_train.T
X_test = X_test.T
y_train = y_train.reshape(1,-1)
y_test = y_test.reshape(1,-1)

In [23]:
nn_arch = [{'layer_units':30,'activation':'none'},
           {'layer_units' : 5, 'activation' : 'relu'},
           {'layer_units' : 4, 'activation': 'relu'},
           {'layer_units' : 3, 'activation': 'relu'},
           {'layer_units' : 1 , 'activation': 'sigmoid'}
           ]

In [33]:
def initialize_parameters(nn_arch,seed_value=42):
    np.random.seed(seed_value)
    parameters = {}
    num_layer = len(nn_arch)
    for l in range(1,num_layer):
        parameters['W' + str(l)]= np.random.randn(
            nn_arch[l]['layer_units'],nn_arch[l-1]['layer_units']
        )*0.01
        parameters['B'+str(l)] = np.zeros((nn_arch[l]['layer_units'],1))
    return parameters

In [34]:
parameters = initialize_parameters(nn_arch,3)

In [35]:
parameters['W1'].shape

(5, 30)

In [36]:
def relu(z):
    return np.maximum(0,z)
def sigmoid(z):
    return 1 / (1+np.exp(-z))

In [37]:
def forward_propagation(nn_arch,x,parameters):
    forward_cache = {}
    num_layers = len(nn_arch)
    forward_cache['A0'] = x
    for l in range(1,num_layers):
        W = parameters['W'+str(l)]
        B = parameters['B'+str(l)]
        activation = nn_arch[l]['activation']
        forward_cache['Z'+str(l)] = np.dot(W,forward_cache['A'+str(l-1)])+B
        if activation=='relu':
           forward_cache['A'+str(l)] = relu(forward_cache['Z'+str(l)])
        elif activation=='sigmoid':
            forward_cache['A'+str(l)] = sigmoid(forward_cache['Z'+str(l)])
    AL =forward_cache['A' + str(num_layers-1)]
    return AL,forward_cache

In [43]:
AL,forward_prop=forward_propagation(nn_arch,X_train,parameters)

In [46]:
def compute_cost(y,AL):
    n = y.shape[1]
    cost = (-1/n)*np.sum((y*np.log(AL)+(1-y)*np.log(1-AL)))
    return cost

In [47]:
compute_cost(y_train,AL)


np.float64(0.6931458921235033)

In [48]:
X_train.shape

(30, 455)

In [49]:
def sigmoid_backward(dA_prev,Z):
    dact = sigmoid(Z)*(1-sigmoid(Z))
    return dA_prev*dact

In [50]:
def relu_backward(dA_prev,Z):
    dA_prev = np.array(dA_prev,copy=True)
    dA_prev[Z<0] = 0
    return dA_prev

In [51]:
def backward_propagation(nn_arch,y,AL,forward_cache,parameters):
    grads = {}
    num_layers = len(nn_arch)
    n = y.shape[1]
    dA_prev = (y - AL) / AL*(1-AL)
    for l in reversed(range(1,num_layers)):
        Z_curr = forward_cache['Z'+str(l)]
        A_prev = forward_cache['A'+str(l-1)]
        W_curr = parameters['W'+str(l)]
        activation = nn_arch[l]['activation']
        if activation == 'relu':
            dZ=1/n*relu_backward(dA_prev,Z_curr)
            grads['dW'+str(l)]= np.dot(dZ,A_prev.T)
            grads['dB' + str(l)] = np.sum(dZ,axis=1,keepdims=True)
            dA_prev=np.dot(W_curr.T,dZ)
        elif activation == 'sigmoid':
            dZ=1/n*sigmoid_backward(dA_prev,Z_curr)
            grads['dW'+str(l)]= np.dot(dZ,A_prev.T)
            grads['dB' + str(l)] = np.sum(dZ,axis=1,keepdims=True)
            dA_prev=np.dot(W_curr.T,dZ)
    return grads

In [55]:
grads = backward_propagation(nn_arch,y_train,AL,forward_prop,parameters)

In [56]:
def update_parameters(parameters,grads,lr,nn_arch):
    for l in range(1,len(nn_arch)):
        parameters['W'+str(l)] = parameters['W'+str(l)] -lr*grads['dW'+str(l)]
        parameters['B'+str(l)] = parameters['B'+str(l)] -lr*grads['dB'+str(l)]
    return parameters

In [57]:
update_parameters(parameters,grads,0.01,nn_arch)

{'W1': array([[ 0.01788628,  0.0043651 ,  0.00096497, -0.01863493, -0.00277388,
         -0.00354759, -0.00082741, -0.00627001, -0.00043818, -0.00477218,
         -0.01313865,  0.00884622,  0.00881318,  0.01709573,  0.00050034,
         -0.00404677, -0.0054536 , -0.01546477,  0.00982367, -0.01101068,
         -0.01185047, -0.0020565 ,  0.01486148,  0.00236716, -0.01023785,
         -0.00712993,  0.00625245, -0.00160513, -0.00768836, -0.00230031],
        [ 0.00745056,  0.01976111, -0.01244123, -0.00626417, -0.00803766,
         -0.02419083, -0.00923792, -0.01023876,  0.01123978, -0.00131914,
         -0.01623285,  0.00646675, -0.00356271, -0.01743141, -0.0059665 ,
         -0.00588594, -0.00873882,  0.00029714, -0.02248258, -0.00267762,
          0.01013183,  0.00852798,  0.01108187,  0.01119391,  0.01487543,
         -0.01118301,  0.00845833, -0.0186089 , -0.00602885, -0.01914472],
        [ 0.01048148,  0.01333738, -0.00197415,  0.01774645, -0.00674728,
          0.00150617,  0.00152

In [62]:
def training(x,y,nn_arch,lr,iterations):
    parameters=initialize_parameters(nn_arch,42)
    costs =[]
    for i in range(iterations):
       AL,forward_cache= forward_propagation(nn_arch, x, parameters)
       cost=compute_cost(y,AL)
       grads=backward_propagation(nn_arch, y, AL, forward_cache, parameters)
       parameters=update_parameters(parameters, grads, lr, nn_arch)
       if(i%100 == 0):
           print('iteration',str(i)+'\n cost : '+str(cost))
           costs.append(cost)
    return parameters,AL

In [64]:
training(X_train,y_train,nn_arch,0.01,1000)

iteration 0
 cost : 0.6931472408795927
iteration 100
 cost : 0.6976251538983106
iteration 200
 cost : 0.7029015560022956
iteration 300
 cost : 0.7091666707791654
iteration 400
 cost : 0.7166695585356444
iteration 500
 cost : 0.7257408366899423
iteration 600
 cost : 0.736825918439747
iteration 700
 cost : 0.7505343442133354
iteration 800
 cost : 0.7677139329208222
iteration 900
 cost : 0.7895632342436162


({'W1': array([[ 4.96714153e-03, -1.38264301e-03,  6.47688538e-03,
           1.52302986e-02, -2.34153375e-03, -2.34136957e-03,
           1.57921282e-02,  7.67434729e-03, -4.69474386e-03,
           5.42560044e-03, -4.63417693e-03, -4.65729754e-03,
           2.41962272e-03, -1.91328024e-02, -1.72491783e-02,
          -5.62287529e-03, -1.01283112e-02,  3.14247333e-03,
          -9.08024076e-03, -1.41230370e-02,  1.46564877e-02,
          -2.25776300e-03,  6.75282047e-04, -1.42474819e-02,
          -5.44382725e-03,  1.10922590e-03, -1.15099358e-02,
           3.75698018e-03, -6.00638690e-03, -2.91693750e-03],
         [-6.01706612e-03,  1.85227818e-02, -1.34972247e-04,
          -1.05771093e-02,  8.22544912e-03, -1.22084365e-02,
           2.08863595e-03, -1.95967012e-02, -1.32818605e-02,
           1.96861236e-03,  7.38466580e-03,  1.71368281e-03,
          -1.15648282e-03, -3.01103696e-03, -1.47852199e-02,
          -7.19844208e-03, -4.60638771e-03,  1.05712223e-02,
           3.4361